# Занятие 10, пара 2. Экспертные оценки и паспорт датасета


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Что сегодня делаем

В первой паре мы размечали отдельные объекты. Теперь посмотрим на решения о **всём наборе**:
что собирать сначала, кого выборка представляет и как передать данные человеку без нашего устного рассказа.
Нужны таблицы, массивы и группировка; новые действия разберём перед применением.
Результат — расчёт согласованности рангов, воспроизводимая выборка и небольшой паспорт датасета.
В конце выдаётся ЛР5, 10 баллов по уже установленным критериям.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


## 1. Три человека выбирают, что улучшать первым

Есть четыре возможных улучшения приложения: поиск, экспорт, уведомления и тёмная тема.
Каждый эксперт ставит их на места 1–4: 1 — самый высокий приоритет. Это **ранги**, не баллы пользы.
Разница между первым и вторым местом не обязана означать столько же, сколько между третьим и четвёртым.

Сначала попросим независимые ответы и один общий вопрос: «Что первым уменьшит число обращений в поддержку?»
Если один ранжирует красоту, другой стоимость, третий скорость разработки, усреднение смешает разные задачи.
Вымышленные ответы ниже нужны для расчёта; это не опрос нашей группы.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ranks = np.array([[1,2,3,4], [1,3,2,4], [2,1,3,4]], dtype=float)
features = ['Поиск', 'Экспорт', 'Уведомления', 'Тёмная тема']
display(pd.DataFrame(ranks, columns=features, index=['A','B','C']))
rank_sum = ranks.sum(axis=0)
display(pd.DataFrame({'feature': features, 'rank_sum': rank_sum}).sort_values('rank_sum'))


Суммирование по `axis=0` складывает голоса для одного улучшения. Меньшая сумма — выше общий приоритет.
Но итоговый порядок скрывает спор. Посчитаем, насколько согласованы **полные ранжирования**.

Коэффициент конкордации Кендалла $W$ сравнивает разброс сумм рангов с максимально возможным:

$$W=\frac{12S}{m^2(n^3-n)},\quad S=\sum_j(R_j-\bar R)^2.$$

$m$ — число экспертов (строк), $n$ — число объектов (столбцов), $R_j$ — сумма рангов объекта.
Если все ставят один порядок, суммы сильно различаются и W=1. Если суммарные ранги одинаковы, W=0.
**Эта формула без поправки допустима только без ничьих**, с полным порядком 1…n у каждого.
Это не каппа из первой пары и не вероятность, что эксперты правы.


In [ ]:
m, n_items = ranks.shape
mean_sum = rank_sum.mean()
deviations = rank_sum - mean_sum
squared_sum = (deviations ** 2).sum()
max_scale = m ** 2 * (n_items ** 3 - n_items)
w = 12 * squared_sum / max_scale
print('Суммы:', rank_sum, 'Среднее:', mean_sum)
print('Квадраты отклонений:', deviations ** 2, 'W:', w)


Получилось около 0.778: в этом маленьком примере порядки близки. Проверим крайний случай,
а не будем просто доверять формуле. Три одинаковых ряда должны дать 1. Обратный порядок одного
эксперта уменьшит согласованность. Новая функция ниже — только упаковка уже разобранных операций.
`np.sort(..., axis=1)` сортирует каждый ряд; сравнение с 1…n проверяет отсутствие пропусков и ничьих.


In [ ]:
def kendall_w_no_ties(values):
    values = np.asarray(values, dtype=float)
    m, n = values.shape
    expected_ranks = np.arange(1, n + 1)
    valid = m >= 2 and n >= 2 and (np.sort(values, axis=1) == expected_ranks).all()
    if not valid:
        return None
    sums = values.sum(axis=0)
    s = ((sums - sums.mean()) ** 2).sum()
    return 12 * s / (m ** 2 * (n ** 3 - n))

print('Одинаковый порядок:', kendall_w_no_ties([[1,2,3],[1,2,3],[1,2,3]]))
print('Противоположные:', kendall_w_no_ties([[1,2,3],[3,2,1]]))


### ✏️ Ваш ход — Очень уверенный, но неверный отчёт

Коллега подставил ранги [[1,2.5,2.5,4], [1,2,3,4]] в формулу
без поправки и записал «эксперты почти согласны, значит выбор оптимален». Получите `my_w` нашей функцией.
В списке `my_claims` оставьте только обоснованные утверждения из номеров:
1 — здесь есть ничья; 2 — наша формула применима; 3 — высокая согласованность доказывает оптимальность;
4 — нужен метод с поправкой на ничьи либо новая независимая процедура ранжирования.
Не разрывайте ничью произвольно ради красивого числа. Объясните две ошибки отчёта.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.2.1', my_w, my_claims)


**Мой вывод:** …


## 2. Из каких объектов состоит выборка

Пусть есть 100 учебных обращений: 90 обычных и 10 редких. Хотим вручную проверить 10.
**Случайная выборка** даёт каждому объекту одинаковый шанс попасть в проверку, но не гарантирует
появление каждой категории в маленькой выборке. `sample(n=10, random_state=...)` выбирает строки
без возвращения. `random_state` фиксирует генератор для воспроизводимости; не улучшает качество сам по себе.
Набор ниже искусственный, его пропорции известны по построению.


In [ ]:
population = pd.DataFrame({
    'item_id': np.arange(100),
    'category': ['ordinary'] * 90 + ['rare'] * 10,
})
random_sample = population.sample(n=10, random_state=7)
print(random_sample['category'].value_counts())


**Стратификация** — сначала делим объекты на известные группы (страты), затем выбираем внутри каждой.
Пропорциональная выборка 9+1 сохраняет заданные доли 90/10. Проверочная 5+5 намеренно чаще показывает
редкий случай. Обе могут быть полезны, но отвечают на разные вопросы.

Сделаем 5+5: это проверка поведения на двух категориях, не оценка их естественной частоты.
`pd.concat(parts, ignore_index=True)` соединит таблицы строками и создаст новый индекс.
Такой отбор возможен, только если категория уже известна; не выдаём её за результат будущей разметки.


In [ ]:
parts = []
for category in ['ordinary', 'rare']:
    group = population.loc[population['category'] == category]
    selected = group.sample(n=5, random_state=7)
    parts.append(selected)
balanced_sample = pd.concat(parts, ignore_index=True)
print(balanced_sample['category'].value_counts())


### ✏️ Ваш ход — Два отчёта из одной проверки

В проверочной выборке 5 обычных и 5 редких обращений.
Модель ошиблась на 1 обычном и 4 редких. Посчитайте `my_balanced_error` — долю ошибок в этой выборке.
Затем `my_population_error` — оценку для известных долей 90%/10%, используя отдельные частоты ошибок.
При каких предположениях второй расчёт переносится на поток? Почему 50% редких в проверке не доказывает
50% редких в потоке?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.2.2', my_balanced_error, my_population_error)


**Мой вывод:** …


## 3. Разделить строки — ещё не разделить информацию

В ML модель подбирает параметры по train. Validation помогает выбрать настройки; test оставляем
для итоговой проверки уже выбранного решения. Если один и тот же разговор попал в train и test
разными сообщениями, модель может узнать знакомый шаблон вместо обобщения на новый разговор.
Такое попадание недоступной при реальном прогнозе информации называется утечкой.

Для связанных объектов вводим group_id: человек, документ, сеанс съёмки или диалог — зависит от задачи.
Ниже одна группа g1 попала в две части. `nunique()` считает число различных значений:
если у одной группы больше одной части, разбиение надо проверить. Само отсутствие пересечений
ещё не доказывает полную независимость: group_id должен быть выбран по смыслу.


In [ ]:
split_example = pd.DataFrame({
    'id': ['a','b','c','d'], 'group_id': ['g1','g1','g2','g3'],
    'split': ['train','test','train','test'],
})
parts_per_group = split_example.groupby('group_id')['split'].nunique()
print('Группы в нескольких частях:', parts_per_group[parts_per_group > 1].index.tolist())


## 4. Паспорт: ответить до того, как коллега спросит

README обычно объясняет запуск проекта. **Паспорт датасета** объясняет сами данные: единицу наблюдения,
происхождение, поля, правила, пропуски, допустимое использование и ограничения.
Не нужен документ на сто страниц. Нужны конкретные ответы, которые нельзя вывести из названия CSV.
JSON удобен для вложенных полей; Markdown удобен для чтения. Выберите один основной вариант.

Пример ниже описывает именно наши **сочинённые** обращения. Мы не приписываем им репрезентативность
реального сервиса. Число строк считаем по файлу. Версия инструкции и происхождение учебных ответов
важнее обещания «данные качественные». Бюджет — отдельно, по допущениям из 09.2.


In [ ]:
import json
import hashlib

source_path = DATA_DIR / 'annotation' / 'messages.csv'
messages = pd.read_csv(source_path)
card = {
    'name': 'Учебные обращения', 'version': '1.0',
    'unit': 'Одно вымышленное короткое обращение',
    'source': 'Сочинено для практики; не выгрузка реального сервиса',
    'rows': len(messages),
    'fields': {
        'item_id': 'Уникальный ключ обращения', 'text': 'Текст для классификации',
        'source_group': 'Условная тройка сценариев; не реальный пользователь',
    },
    'annotation': 'Инструкция v1: issue/request/question/unclear; пусто = нет ответа',
    'example_labels': 'Два вымышленных ряда, не золотой стандарт и не ответы группы',
    'allowed_use': 'Учебная практика с указанием происхождения',
    'limitations': ['Малая сочинённая выборка', 'Не измеряет частоты реальных запросов',
                    'Нет оснований оценивать качество на настоящих пользователях'],
    'split': 'Разбиение для обучения не подготовлено; source_group не доказывает независимость',
}
output_dir = WORKSPACE_DIR / 'outputs' / 'dataset_documentation'
output_dir.mkdir(parents=True, exist_ok=True)
card_path = output_dir / 'dataset_card.json'
card_path.write_text(json.dumps(card, ensure_ascii=False, indent=2), encoding='utf-8')
print(card_path)
display(card)


Новая служебная операция `hashlib.sha256(bytes).hexdigest()` строит отпечаток **содержимого файла**.
`read_bytes()` читает байты. Изменили файл — отпечаток почти наверняка другой; это помогает точно
назвать использованную версию. Хеш не доказывает качество, авторство или отсутствие персональных данных.
Не используйте его как способ обезличивания: об этом следующий блок.


In [ ]:
content_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
version_record = {'file': source_path.name, 'sha256': content_hash, 'guideline': 'v1'}
(output_dir / 'data_version.json').write_text(
    json.dumps(version_record, ensure_ascii=False, indent=2), encoding='utf-8')
print(version_record)


### ✏️ Ваш ход — Паспорт с рекламой вместо фактов

В чужом паспорте написано: «30 реальных обращений, всё проверено;
согласие 100%, поэтому ошибок нет; можно обучать промышленный классификатор; источник — интернет».
Создайте `my_fixes` — словарь с полями source, agreement, use. В каждом поле напишите конкретное
исправление для **нашего** учебного набора. Не просто удалите спорную фразу: замените проверяемым описанием.
Проверка проверит только заполненность; смысл обсудите и сверяйте с SOURCE.md.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.2.3', my_fixes)


**Мой вывод:** …


## ЛР5 — разметка и документация, 10 баллов

Полное задание — `docs/ЛР5.md`: откройте папку docs в корне своей копии проекта.
Разметка/спорные случаи — 3; согласованность и смысл — 3; паспорт — 2; оценка затрат — 2.
Можно работать вне пары, сдавать файлами или архивом. GitHub необязателен.
Каппа Флейса, W, DVC и наличие второго аккаунта не становятся дополнительными критериями.

Результат этой пары: dataset_card.json и data_version.json — примеры для своего проекта.
Не выдавайте их автоматически за паспорт собственного набора: перепишите происхождение и ограничения.
В следующем блоке отдельно исследуем персональные данные и обезличивание.


## Резерв: как сильно результат зависит от выборки?

Повторите случайную проверку для пяти значений seed, выпишите число редких объектов.
Сравните с отбором 9+1 и 5+5. Выбирайте схему по вопросу проверки, а не seed с удобным результатом.
Для своего проекта укажите, что является независимым объектом: строка, человек или целый документ.

При росте объёма [DVC](https://dvc.org/doc/start/data-management/data-versioning) связывает версии данных
с версиями проекта; на этой паре достаточно файла, паспорта и отпечатка. Установка не нужна.
Идея паспорта и вопросы для обсуждения: [Datasheets for Datasets](https://arxiv.org/abs/1803.09010).
Это ориентир для описания данных, не требование заполнить все вопросы большой статьи.
Определение W и поправка на ничьи: [документация пакета irr](https://search.r-project.org/CRAN/refmans/irr/html/kendall.html).
